# Data-Driven Chord-to-chord Markhov Model

In [ ]:
import numpy as np
import os
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
import mm_part_to_part as p2p

In [ ]:
def get_chord_dict(dataset_dir):
    """
    Data-driven approach to enumerate all observed chords

    Args:
        dataset_dir (str): The directory containing the training data.

    Returns:
        dict: dictionary encoding (A, T, B) --> idx 
        chord_to_chord: max(idx)xmax(idx) state transition matrix for chords
    """
    songs = glob.glob('*.csv', root_dir=dataset_dir)
    songs = [s for s in songs if not 'd' in s] #remove all strings with 'd' in them (filters out chord csv)
    chord_to_idx = {}
    idx_to_chord = {}
    chord_idx_list = []
    idx = 0

    #can can shrink because they don't have access to all notes, just going to make a 128x128 mat
    counts_A = np.ones((128, 128), dtype='int64') *1/1000 #add an epsilon so no divide by zero
    counts_T = np.ones((128, 128), dtype='int64') *1/1000 #add an epsilon so no divide by zero
    counts_B = np.ones((128, 128), dtype='int64') *1/1000 #add an epsilon so no divide by zero

    for song in tqdm(songs, desc="Parsing CSVs", unit="song"):
        song_path = os.path.join(dataset_dir, song)

        if os.path.exists(song_path):
            try:
                _, A, T, B = p2p.csv_to_tracks(song_path)

            except Exception as e:
                tqdm.write(f"Error processing {song_path}: {e}")

            N = len(B) #how many datapoints we have
            ATB = []
            for i in range(1, N):
                chord = (A[i], T[i], B[i])
                # make array of chords in order,  then make dict and use for transition model
                # question.. big array okay? like 100x100
                if chord not in chord_to_idx:
                    chord_to_idx[chord] = idx
                    idx_to_chord[idx] = chord
                    idx += 1
                chord_idx_list.append(chord_to_idx[chord]) # add chord idx to list of seen chords   
    chord_to_chord = np.ones((idx, idx), dtype='int64')*1/1000 # add an epsilon so no divide by zero
    # for i in range(idx):
    #      chord_to_chord[]
    for i in range(1,chord_idx_list):  #index thru all chords seen
        prev_chord_idx = chord_idx_list[i-1]
        chord_idx = chord_idx_list[i]
        chord_to_chord[prev_chord_idx, chord_idx] += 1
         # construct!
                    # counts_A[int(A[i-1]), int(A[i])] += 1
                    # counts_T[int(T[i-1]), int(T[i])] += 1
                    # counts_B[int(B[i-1]), int(B[i])] += 1

In [ ]:
train_dir = os.path.join('jsb_chorales', 'train')
